## 7. N-HiTS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 62.2 MB/s eta 0:00:00
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import NHiTSPredictor, HORIZONS

MODEL_NAME = 'NHiTS'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'nhits'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'nhits_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [4]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = NHiTSPredictor(context_length=sample_cgm.shape[1])

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'nhits' /f'nhits_seed{seed}',
        verbose=False,
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'nhits_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — NHiTS | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: NHiTSPredictor  params: 70,002


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.11 | MAE 7.48 | gRMSE 12.26 | Zone A 97.08% | A+B 99.83%
 30 min | RMSE 18.40 | MAE 12.21 | gRMSE 21.32 | Zone A 90.66% | A+B 99.56%
 60 min | RMSE 27.15 | MAE 17.96 | gRMSE 33.12 | Zone A 81.99% | A+B 99.09%
 90 min | RMSE 32.39 | MAE 21.78 | gRMSE 40.24 | Zone A 75.54% | A+B 98.59%
120 min | RMSE 35.34 | MAE 24.04 | gRMSE 44.29 | Zone A 71.32% | A+B 98.24%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: NHiTSPredictor  params: 70,002


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.10 | MAE 7.47 | gRMSE 12.09 | Zone A 96.95% | A+B 99.84%
 30 min | RMSE 18.30 | MAE 12.23 | gRMSE 20.90 | Zone A 90.57% | A+B 99.61%
 60 min | RMSE 26.90 | MAE 18.04 | gRMSE 32.47 | Zone A 81.82% | A+B 99.15%
 90 min | RMSE 32.18 | MAE 21.82 | gRMSE 39.74 | Zone A 75.28% | A+B 98.59%
120 min | RMSE 35.21 | MAE 24.01 | gRMSE 44.01 | Zone A 71.23% | A+B 98.28%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: NHiTSPredictor  params: 70,002


...

 15 min | RMSE 11.04 | MAE 7.41 | gRMSE 12.17 | Zone A 97.22% | A+B 99.83%
 30 min | RMSE 18.30 | MAE 12.02 | gRMSE 21.15 | Zone A 90.95% | A+B 99.56%
 60 min | RMSE 27.02 | MAE 17.60 | gRMSE 32.91 | Zone A 82.54% | A+B 99.07%
 90 min | RMSE 32.23 | MAE 21.29 | gRMSE 40.08 | Zone A 76.56% | A+B 98.57%
120 min | RMSE 35.16 | MAE 23.48 | gRMSE 44.17 | Zone A 72.98% | A+B 98.21%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: NHiTSPredictor  params: 70,002


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.19 | MAE 7.50 | gRMSE 12.50 | Zone A 97.22% | A+B 99.82%
 30 min | RMSE 18.40 | MAE 12.16 | gRMSE 21.44 | Zone A 90.73% | A+B 99.55%
 60 min | RMSE 27.05 | MAE 17.88 | gRMSE 33.05 | Zone A 82.18% | A+B 99.08%
 90 min | RMSE 32.26 | MAE 21.53 | gRMSE 40.17 | Zone A 75.92% | A+B 98.49%
120 min | RMSE 35.20 | MAE 23.76 | gRMSE 44.19 | Zone A 72.14% | A+B 98.21%

--------------------------------------------------------------------------------
Test results — NHiTS | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: NHiTSPredictor  params: 70,002


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.02 | MAE 7.39 | gRMSE 12.13 | Zone A 97.30% | A+B 99.86%
 30 min | RMSE 18.25 | MAE 12.04 | gRMSE 21.07 | Zone A 90.67% | A+B 99.61%
 60 min | RMSE 26.97 | MAE 17.75 | gRMSE 32.83 | Zone A 81.89% | A+B 99.13%
 90 min | RMSE 32.21 | MAE 21.51 | gRMSE 40.02 | Zone A 75.92% | A+B 98.60%
120 min | RMSE 35.23 | MAE 23.80 | gRMSE 44.20 | Zone A 71.78% | A+B 98.18%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )


=== NHiTS test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.11 +/- 0.08 (1.81) mg/dL   MAE = 7.46 +/- 0.06 (1.34)   gRMSE = 12.23 +/- 0.13 (2.86)   Zone A = 97.20 +/- 0.13 (1.86)%   Zone A+B = 99.85 +/- 0.03 (0.21)%
 30-min:  RMSE = 18.34 +/- 0.07 (3.72) mg/dL   MAE = 12.12 +/- 0.08 (2.45)   gRMSE = 21.17 +/- 0.17 (6.17)   Zone A = 90.75 +/- 0.11 (3.01)%   Zone A+B = 99.59 +/- 0.03 (0.46)%
 60-min:  RMSE = 27.02 +/- 0.07 (8.67) mg/dL   MAE = 17.84 +/- 0.15 (5.25)   gRMSE = 32.87 +/- 0.18 (13.69)   Zone A = 82.14 +/- 0.27 (4.59)%   Zone A+B = 99.11 +/- 0.04 (0.85)%
 90-min:  RMSE = 32.25 +/- 0.07 (12.38) mg/dL   MAE = 21.55 +/- 0.19 (7.45)   gRMSE = 40.06 +/- 0.18 (19.32)   Zone A = 75.95 +/- 0.43 (6.09)%   Zone A+B = 98.58 +/- 0.04 (1.44)%
120-min:  RMSE = 35.23 +/- 0.09 (14.75) mg/dL   MAE = 23.80 +/- 0.19 (9.10)   gRMSE = 44.19 +/- 0.20 (22.93)   Zone A = 71.95 +/- 0.59 (6.43)%   Zone A+B = 98.23 +/- 0.07 (2.13)%


In [7]:
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/nhits_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nhits/nhits_seed*_results.json
saved aggregate summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/nhits/nhits_stability_summary.json
